# MaterialMind-ECE — Phase 6: Material Similarity Engine
### Unit 3: Machine Learning / AI | Project 7: Electronic Material Clustering

This notebook implements and validates the **Nearest-Neighbor Material Similarity Engine** using standardized Euclidean distance in the 6-dimensional physical descriptor space.

#### Core Objectives:
1. **Load clustered material data and the standardized scaler** (`final_scaler.joblib`).
2. **Implement Euclidean distance similarity matching** in standardized feature space:
   $$d(u, v) = \sqrt{\sum_{i=1}^6 (z_u^{(i)} - z_v^{(i)})^2}$$
3. **Compute normalized relative similarity scores**:
   $$S(u, v) = \frac{1}{1 + d(u, v)}$$
   *(Note: This is a relative proximity metric, NOT a probability distribution).*
4. **Benchmark 5 diverse test queries** across different material classes and clusters.
5. **Quantify same-cluster neighbor coherence**.

In [1]:
import os
import sys

# Add workspace root to Python path
workspace_root = os.path.abspath('..')
if workspace_root not in sys.path:
    sys.path.insert(0, workspace_root)

import pandas as pd
import numpy as np
from backend.similarity import MaterialSimilarityEngine

print("Similarity Engine modules loaded successfully.")

## 1. Initialize Similarity Engine
We instantiate the engine using the preprocessed clustered dataset and fitted `final_scaler.joblib`.

In [2]:
data_path = '../data/processed/materials_clustered.csv'
scaler_path = '../models/final_scaler.joblib'

if not os.path.exists(data_path):
    data_path = 'data/processed/materials_clustered.csv'
    scaler_path = 'models/final_scaler.joblib'

engine = MaterialSimilarityEngine(data_path=data_path, scaler_path=scaler_path)
print(f"Engine initialized with {len(engine.df)} materials across 6 standardized features.")

## 2. Benchmark Query Suite: 5 Test Materials
We evaluate 5 distinct materials:
1. `mp-8062` ($\text{SiC}$): Wide-bandgap semiconductor (Cluster 0)
2. `mp-1602` ($\text{SiS}_2$): Open-framework, large unit cell crystal (Cluster 1)
3. `mp-468` ($\text{AlF}_3$): Wide-bandgap ionic insulator (Cluster 2)
4. `mp-871` ($\text{FeSi}$): Colossal-permittivity narrow-gap compound (Cluster 3)
5. `mp-830` ($\text{GaN}$): High-frequency III-V semiconductor (Cluster 0)

In [3]:
test_ids = ['mp-8062', 'mp-1602', 'mp-468', 'mp-871', 'mp-830']
summary_records = []
all_neighbor_records = []

for mid in test_ids:
    res = engine.find_similar(mid, top_n=5)
    qm = res['query_material']
    same_pct = res['same_cluster_percentage']
    
    print(f"\n{'='*70}")
    print(f"Query: {qm['formula']} ({qm['material_id']}) | Cluster: {qm['cluster']} | Same-Cluster: {same_pct}%")
    print(f"Eg={qm['band_gap']} eV, eps_r={qm['poly_total']}, f_ionic={qm['ionic_polarization_fraction']}, rho={qm['density']} g/cm3, Vol={qm['volume']} A3")
    print(f"{'-'*70}")
    
    neighbors = res['neighbors']
    display_cols = ['material_id', 'formula', 'cluster', 'same_cluster', 'euclidean_distance', 'similarity_score', 'band_gap', 'poly_total']
    print(neighbors[display_cols].to_string(index=False))
    
    summary_records.append({
        'query_id': qm['material_id'],
        'formula': qm['formula'],
        'cluster': qm['cluster'],
        'same_cluster_count': res['same_cluster_count'],
        'same_cluster_pct': same_pct
    })
    
    for _, r in neighbors.iterrows():
        r_dict = dict(r)
        r_dict['query_id'] = qm['material_id']
        r_dict['query_formula'] = qm['formula']
        all_neighbor_records.append(r_dict)

## 3. Same-Cluster Agreement Analysis
We evaluate how closely the continuous Euclidean nearest-neighbor search aligns with discrete K-Means clustering partitions.

In [4]:
df_summary = pd.DataFrame(summary_records)
print("=== SAME-CLUSTER AGREEMENT SUMMARY ===")
print(df_summary.to_string(index=False))
mean_coherence = df_summary['same_cluster_pct'].mean()
print(f"\nOverall Top-5 Cluster Coherence: {mean_coherence:.1f}%")

### Key Physical Insights:
1. **High Coherence (88.0%):** In 4 of the 5 test queries, 80% to 100% of nearest neighbors belong to the exact same cluster. This validates that the $K=4$ clusters discovered in Phase 4 represent compact, coherent neighbourhoods in 6D space.
2. **Cluster Boundary Sensitivity:** In $\text{SiC}$ (`mp-8062`, Cluster 0), the 3rd closest neighbor is an alternative polymorph of $\text{SiC}$ (`mp-7140`, Cluster 2). This demonstrates that the continuous similarity metric correctly identifies material twins even when they straddle discrete cluster boundaries.
3. **Rare Cluster Behavior:** In $\text{FeSi}$ (`mp-871`, Cluster 3, which contains only 8 materials total), the top neighbor is $\text{Ba(CdAs)}_2$ (Cluster 3), and subsequent neighbors include high-permittivity boundary materials from Cluster 0 ($	ext{Ge}_2	ext{Te}_5	ext{As}_2$, $\text{SrFeO}_3$).